# A Pythonic Object

## Core Idea

Chapter 1 showed the basics of special methods. Chapter 11 goes deeper: 
what does it take to make a class feel like a genuinely well-designed, 
"Pythonic" object — not just a class that technically works? 

The chapter builds this up through one running example: a `Vector2d` class, 
progressively adding: multiple string representations, an alternate 
constructor, custom formatting, read-only attributes (immutability), and 
hashability.

## The four representation dunders

| Method | Called by | Purpose |
|---|---|---|
| `__repr__` | `repr(obj)`, console/debugger display | for developers — should look like the code needed to recreate the object |
| `__str__` | `str(obj)`, `print(obj)` | for end users — human-friendly display |
| `__bytes__` | `bytes(obj)` | binary representation |
| `__format__` | f-strings, `format(obj)`, `.format()` | custom formatting codes |

**Rule of thumb**: if you only implement one, implement `__repr__` — Python 
falls back to it for `str()` if `__str__` is missing.

**Why `__iter__` matters here**: making `Vector2d` iterable is what enables 
`x, y = v1` (unpacking) AND lets `__repr__`/`__str__` use `*self` and 
`tuple(self)` cleanly — one small addition (`__iter__`) pays off in several 
other methods "for free," same idea as `FrenchDeck` in Ch. 1.

**Known limitation worth noting**: this `__eq__` also returns `True` when 
comparing a `Vector2d` to any other iterable with matching values (e.g. 
`Vector2d(3,4) == [3,4]` → `True`). This may or may not be desired — the 
book flags it as a real design tradeoff, revisited later in the operator 
overloading chapter.

In [ ]:
# Object representations — the four ways to represent an object

vfrom array import array
import math

class Vector2d:
    typecode = 'd'   # class attribute, used when converting to/from bytes

    def __init__(self, x, y):
        self.x = float(x)   # cast early -> catches bad input immediately
        self.y = float(y)

    def __iter__(self):
        return (i for i in (self.x, self.y))   # makes the object iterable -> unpacking works

    def __repr__(self):
        class_name = type(self).__name__
        return '{}({!r}, {!r})'.format(class_name, *self)

    def __str__(self):
        return str(tuple(self))

    def __bytes__(self):
        return (bytes([ord(self.typecode)]) +
                bytes(array(self.typecode, self)))

    def __eq__(self, other):
        return tuple(self) == tuple(other)

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

## classmethod vs staticmethod

- **`@classmethod`**: receives the **class itself** (`cls`) as the first 
argument, not an instance. Used to build **alternate constructors** — 
ways to create an instance that don't go through the normal `__init__` 
signature. `frombytes` reconstructs a `Vector2d` from binary data, as an 
alternative to `Vector2d(x, y)`.
- **`@staticmethod`**: just a plain function that happens to live inside a 
class namespace — receives neither `self` nor `cls`. Used purely for 
organizational grouping, not for polymorphism.

**Practical rule**: reach for `@classmethod` when you need a second way to 
construct an object (parsing from a string, bytes, a config dict, etc.) — 
this is a genuinely common, useful pattern worth remembering.

In [ ]:
@classmethod
def frombytes(cls, octets):
    typecode = chr(octets[0])
    memv = memoryview(octets[1:]).cast(typecode)
    return cls(*memv)

In [ ]:
# Immutability via read-only properties

class Vector2d:
    typecode = 'd'

    def __init__(self, x, y):
        self.__x = float(x)     # note the double underscore -> name mangling
        self.__y = float(y)

    @property
    def x(self):
        return self.__x

    @property
    def y(self):
        return self.__y

## Making attributes read-only

- Store the real value under a "private" name (`self.__x`), then expose a 
**read-only property** called `x` via `@property` — a method that behaves 
like an attribute when accessed (`v1.x`, not `v1.x()`).
- Since there's no matching `@x.setter`, `v1.x = 123` raises `AttributeError: 
can't set attribute` — the vector becomes effectively immutable after creation.
- Everything else in the class (`__repr__`, `__iter__`, etc.) can keep using 
`self.x` / `self.y` unchanged — the property makes the read-only-ness 
invisible to the rest of the class.

## Private/protected attribute conventions

Python has **no true private variables** (unlike Java's `private`). Two 
conventions exist instead:

- **`self.__x`** (double leading underscore) → Python automatically renames 
this internally to `self._ClassName__x` ("name mangling"). Check it 
yourself: `v1.__dict__` shows `{'_Vector2d__x': 3.0, ...}`. 
  - **Purpose**: prevents *accidental* attribute clashes in subclasses — 
  not real security. Anyone who knows the mangling rule can still access 
  `v1._Vector2d__x` directly.
- **`self._x`** (single leading underscore) → pure convention, no special 
Python behavior at all. Just a signal to other programmers: "don't touch 
this from outside the class." This is the **more common, more idiomatic** 
choice among Python developers — some (like pip's creator, quoted in the 
book) actively discourage double-underscore mangling as unnecessarily 
obscure.

**Takeaway**: "private" and "immutable" in Python are always in scare 
quotes — they're conventions and mild safety nets, not hard guarantees.

## __hash__ and __eq__ — go together

- If you define `__eq__` but not `__hash__`, your objects become 
**unhashable** — you can't put them in a `set` or use them as dict keys. 
This connects directly back to Chapter 3's hashability rules.
- **Convention**: combine the hashes of the object's components with `^` 
(XOR) — this is the standard, recommended pattern for combining multiple 
hash values into one.
- This only makes sense once the object is genuinely immutable (read-only 
properties above) — a mutable object's hash could change after being 
inserted into a set/dict, breaking the hash table, exactly as discussed in 
Chapter 3.

In [ ]:
# __hash__ — making an object usable in sets/dicts

def __hash__(self):
    return hash(self.x) ^ hash(self.y)

# EXERCISES

Build a `Vector2D` class with:
  1. `__init__(self, x, y)`
  2. `__repr__` (should return something like `Vector2D(3, 4)`)
  3. `__eq__` (two vectors equal if x and y match)
  4. `__add__` (vector addition returns a new `Vector2D`)
  5. Test: create two vectors, add them, print the result, and confirm equality comparison works.

In [2]:
class Vector2D:
    def __init__(self,x,y):
        self.x = float(x)
        self.y = float(y)

    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r})'

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        return Vector2D(x, y)

v1 = Vector2D(2,4)
v2 = Vector2D(2,4)

print(v1)          # uses __repr__ since there's no __str__
print(v1 + v2)      # uses __add__
print(v1 == v2)     # uses __eq__
print(v1 == Vector2D(2, 4))   # should be True




Vector(2.0, 4.0)
Vector(4.0, 8.0)
True
True
